# Assignment 2: Metabolic Modeling

## Part1: Visualizing reaction maximal activity data and making some sense of it.

a) No, the maximal reactions are not equal in a linear pathway, for example between PGM and ENO, there's a difference in the maximal reactions. The maximal reaction of PGM is 21.7, while the maximal reaction of ENO is 29.3. this is because the maximal reactions is the vmax, in other words the maximal number of reactions that can occur depending on the enzyme availability, and this might be different for adjacent enzymes in a linear pathway.

b) The two possible values that the grey arrows can have are 0.00 or n.d. The difference is that 0.00 means that data for this reaction was collected and it was 0.00, which might mean that that gene is not expressed in the cell. While n.d. means "no data", which moight mean that the data for this reaction is not in the loaded dataset.

## Part2: Adjusting upper and lower bounds.

In [2]:
# Import for libraries used (with installatioin)
import cobra
import csv

In [3]:
# loading the ecoli model taken from the computer practical.
model = cobra.io.load_json_model('e_coli_core-1.json')

In [4]:
# csv file with reaction activity
filename = "KEN3170_Assignment_2026_e_coli_core_expression.csv"

In [5]:
# reading the csv file and saving the values in activity_constraints to be used later
activity_constraints = {}

with open(filename, mode="r", newline="") as file:
    reader = csv.reader(file)

    for row in reader:
        if not row or row[0].startswith("#"):
            continue

        reaction_id = row[0]
        value = float(row[1])

        activity_constraints[reaction_id] = value

In [6]:
#setting lower and upper bounds according to instructions provided and with activity data taken from the csv file
for reaction in model.reactions:
    
    reaction_id = reaction.id

    if reaction_id in activity_constraints: #This makes sure we only chnage the bounds for reactions with data

        #we will use this to change bounds to their activity data (when applicable)
        value = activity_constraints[reaction_id]

        # if ATPM energy mainenance
        if reaction_id == "ATPM":
            reaction.upper_bound = value

        # if reversible reaction
        elif reaction.lower_bound < 0:
            reaction.lower_bound = -value
            reaction.upper_bound = value

        #if irreversable reaction
        else:
            reaction.upper_bound = value

    # if glucose exchange reaction
    elif reaction_id == "EX_glc__D_e":
        reaction.lower_bound = -1000
        reaction.upper_bound = 1000
        

In [7]:
#print out for a table with each reaction's lower and uppoer flux bound
print("reaction lowerB upperB")
for reaction in model.reactions:
    print(
        reaction.id,
        reaction.lower_bound,
        reaction.upper_bound
    )

reaction lowerB upperB
PFK 0.0 13.1
PFL 0.0 0.0
PGI -11.1 11.1
PGK -24.0 24.0
PGL 0.0 7.3
ACALD -0.0 0.0
AKGt2r -0.0 0.0
PGM -21.7 21.7
PIt2r -5.2 5.2
ALCD2x -0.0 0.0
ACALDt -1000.0 1000.0
ACKr -2.5 2.5
PPC 0.0 3.6
ACONTa -21.4 21.4
ACONTb -21.4 21.4
ATPM 8.39 1000.0
PPCK 0.0 13.3
ACt2r -3.6 3.6
PPS 0.0 3.1
ADK1 -27.4 27.4
AKGDH 0.0 26.7
ATPS4r -80.1 80.1
PTAr -4.47 4.47
PYK 0.0 28.2
BIOMASS_Ecoli_core_w_GAM 0.0 1000.0
PYRt2 -0.0 0.0
CO2t -1000.0 1000.0
RPE -6.3 6.3
CS 0.0 21.4
RPI -5.6 5.6
SUCCt2_2 0.0 0.0
CYTBD 0.0 41.1
D_LACt2 -0.0 0.0
ENO -29.3 29.3
SUCCt3 0.0 0.0
ETOHt2r -1000.0 1000.0
SUCDi 0.0 27.3
SUCOAS -19.4 19.4
TALA -4.5 4.5
THD2 0.0 6.5
TKT1 -3.5 3.5
TKT2 -3.5 3.5
TPI -70.0 70.0
EX_ac_e 0.0 1000.0
EX_acald_e 0.0 1000.0
EX_akg_e 0.0 1000.0
EX_co2_e -1000.0 1000.0
EX_etoh_e 0.0 1000.0
EX_for_e 0.0 1000.0
EX_fru_e 0.0 1000.0
EX_fum_e 0.0 1000.0
EX_glc__D_e -1000 1000
EX_gln__L_e 0.0 1000.0
EX_glu__L_e 0.0 1000.0
EX_h_e -1000.0 1000.0
EX_h2o_e -1000.0 1000.0
EX_lac__D_e 0.0 10

## Part3: Flux Balance Analysis, absolute flux bound and FBA biomass.

In [8]:
# 3a
# optimize the model for biomass production
solution = model.optimize()

# print maximal biomass production
print("Maximal biomass production:", solution.objective_value)



Maximal biomass production: 0.8732862458582367


In [9]:
# 3b
# setting the absolute flux bound for the glucose exchange reaction lower bound to 5.0
# in cobrapy the flux is defined so that the intake into the cell has a negative sign
model.reactions.get_by_id("EX_glc__D_e").lower_bound = -5.0

3b) this constraint describes an upper limit in the intake rate of glucose into the cell (extracellular). Whereas the other expression-based constraints decribe bounds on the internal reaction rates of the metabolites in the cell (intracellular).

In [10]:
# 3c
# optimize the model for biomass production with the new glucose intake constraint
solution = model.optimize()
print("Maximal biomass production with new glucose intake constraint:", solution.objective_value)

Maximal biomass production with new glucose intake constraint: 0.41559777509290663
